# Project 2
## Bipartite network: User movie rating
### Source: https://grouplens.org/datasets/movielens/100k/
### Author: Chi Hang (Philip) Cheung

Instructions
1. Identify a large 2-node network dataset—you can start with a dataset in a repository.  Your data should meet the criteria that it consists of ties between and not within two (or more) distinct groups.
2. Reduce the size of the network using a method such as the island method described in chapter 4 of social network analysis.
3. What can you infer about each of the distinct groups?

In [13]:
import polars as pl
import networkx as nx
from networkx.algorithms import bipartite
from networkx.algorithms.community import louvain_communities


### 1) Load csv to Polars for more memory efficiency

In [14]:
df = pl.scan_csv(
    'user_rating_history.csv', 
    schema_overrides={"rating": pl.Float64},
    null_values=['N/A', 'NA']
)

processed_edge = (
    df.select(['userId', 'movieId', 'rating'])
    .filter(pl.col('rating') >= 1.0)
    .sort(['movieId', 'userId'], descending=[False, False])
)

# Execute the plan
final_df = processed_edge.collect()

print(final_df.head())

shape: (5, 3)
┌────────┬─────────┬────────┐
│ userId ┆ movieId ┆ rating │
│ ---    ┆ ---     ┆ ---    │
│ i64    ┆ i64     ┆ f64    │
╞════════╪═════════╪════════╡
│ 42170  ┆ 1       ┆ 4.0    │
│ 43715  ┆ 1       ┆ 4.0    │
│ 44282  ┆ 1       ┆ 3.0    │
│ 50108  ┆ 1       ┆ 4.5    │
│ 50602  ┆ 1       ┆ 5.0    │
└────────┴─────────┴────────┘


### 2) To load the df into network X

In [15]:
#Define the graph:
B = nx.Graph()

#Extract User and Movie and add identifier for user with "u_" and movie with "m_" to prevent overlapping IDs:
users = ["u_" + str(uid) for uid in final_df['userId'].unique()]
movies = ["m_" + str(mid) for mid in final_df['movieId'].unique()]

#Add nodes to the graph with their bipartite group labels:
B.add_nodes_from(users, bipartite=0) #Group 0 = users
B.add_nodes_from(movies, bipartite=1) #Group 1 = movies

#Add edges with weights:
edges = [
    ("u_" + str(row['userId']), "m_" + str(row['movieId']), row['rating']) 
    for row in final_df.iter_rows(named=True)
]
B.add_weighted_edges_from(edges)

#Let's check the total number of nodes and edges:
print("Number of nodes:", B.number_of_nodes())
print("Number of edges:", B.number_of_edges())

Number of nodes: 42259
Number of edges: 813971


### 3) Apply the island method to filter out weak connections:

    -Water level/Threshold = rating of 5

In [16]:
#Calculate the original number of nodes:
print(f'Is bipartite: {bipartite.is_bipartite(B)}')
print(f'Total Nodes: {B.number_of_nodes()}')
print(f'Total edges: {B.number_of_edges()}')
print('-'*80)

#Apply Island method. Set threshold = 5 rating for movies:
threshold = 5

strong_edge = [
    (u,v) for u, v, edge_data in B.edges(data=True) if edge_data['weight'] >= threshold
]

strong_islands = B.edge_subgraph(strong_edge).copy() #drops nodes with 0 connections

#Find the remaining connected islands:
islands = list(nx.connected_components(strong_islands))

print(f'Number of distinct island: {len(islands)}')
print(f'Remaining number of nodes after island method: {strong_islands.number_of_nodes()}')
print(f'Remaining number of edges after island method: {strong_islands.number_of_edges()}')
print('-'*80)
print(f'Percentage of reduction for nodes/edges: {100-(strong_islands.number_of_nodes()/B.number_of_nodes() * 100):.2f}% /{100 - (strong_islands.number_of_edges() / B.number_of_edges() * 100):.2f}%')

Is bipartite: True
Total Nodes: 42259
Total edges: 813971
--------------------------------------------------------------------------------
Number of distinct island: 2
Remaining number of nodes after island method: 11259
Remaining number of edges after island method: 63646
--------------------------------------------------------------------------------
Percentage of reduction for nodes/edges: 73.36% /92.18%


By setting the "water level" to rating of 5, the highest possible rating for movies in the dataset, we have effectively filtered out **73.36%** of the nodes and **92.18%** of the edges or connections. Two very distinct islands were revealed. We should examine further in these two islands to see if there is any commonalities between them.

### 4) Two main islands with rating of 5

In [17]:
#Since there are two islands, we can check the size of each of them:
[len(island) for island in islands]

[11257, 2]

We can see that only the first island or island 1 is the main island with 11257 nodes while island 2 is negligible, with only 2 nodes. We Will focus on investigating island 1 only.

In [18]:
island1 = islands[0] #Define the main island

#Check the users who review the most movies:
top_user = []
most_rated_movie = []
for n in island1:
    if n.startswith('u_'):
        top_user.append((strong_islands.degree(n), n))
    elif n.startswith('m_'):
        most_rated_movie.append((strong_islands.degree(n), n))

#Sort both lists in descending order:
top_user_sorted = sorted(top_user, reverse=True)
most_rated_movie_sorted = sorted(most_rated_movie, reverse=True)

#Show top 5 users and movies that are being actively rated with 5:
print(f'Top 5 users who frequently rate movies: {[user for degree, user in top_user_sorted[:5]]}')
print(f'Top 5 movies being rated 5: {[movie for degree, movie in most_rated_movie_sorted[:5]]}')

Top 5 users who frequently rate movies: ['u_125273', 'u_286707', 'u_252975', 'u_55083', 'u_367907']
Top 5 movies being rated 5: ['m_318', 'm_2571', 'm_296', 'm_2959', 'm_58559']


### What are some Blockbuster movies in island 1?

In [19]:
#Separate movies from users:
movies_nodes = [n for n in island1 if n.startswith('m_')]

#Project the bipartite graph into a movie-only graph
#In this graph, two movies are connected if the same user rated BOTH 5 stars
movie_network = bipartite.projected_graph(strong_islands, movies_nodes)

#Check which movie has the highest connections in this projection:
top_connector_movie = sorted(movie_network.degree, key=lambda x: x[1], reverse=True)

print("Movies most frequently co-rated 5-stars with other movies:")
print(top_connector_movie[:10])

Movies most frequently co-rated 5-stars with other movies:
[('m_858', 7038), ('m_296', 6648), ('m_318', 6521), ('m_4993', 6195), ('m_1196', 6068), ('m_5618', 6035), ('m_260', 6026), ('m_924', 6009), ('m_527', 5987), ('m_541', 5977)]


The top_connector_movie returns a the top 10 movies that are frequently being rated with max score with other movies by the same users. In other words, m_858, which has the highest degree, is being loved by so many other users who might have other preferences in genre but still regard this movie highly. According to the metadata, m_858 is "Godfather, The (1972)"  followed by "m_296" Pulp Fiction (1994). These are indeed classic titles that are enjoyed by many viewers.

### 5) To find user "Taste Communities" in Island 1:
#### We want to identify what kind of movie or genres are being subgrouped among the network.

In [20]:
communities = louvain_communities(movie_network, weight='weight')
print(f'Number of Taste Communities found: {len(communities)}')

Number of Taste Communities found: 5


We will explore more in these communities:

In [21]:
#Load the movie metadata into Polars:
movie_df = pl.read_csv('movies.csv')

#Create a mapping for the movie titles:
movie_id_map = {
    f'm_{row['movieId']}': row['title'] for row in movie_df.iter_rows(named=True)
}

movie_genre_map = {
    f'm_{row['movieId']}': row['genres'] for row in movie_df.iter_rows(named=True)
}

#Print the number of movies in these communities:
for i, community in enumerate(sorted(communities, key=len, reverse=True)):
    print(f'\n--- Taste Community {i+1} ({len(community)} movies)')

    #Sort the movies by their degree:
    community_movie_sorted = sorted(list(community), key=lambda x: movie_network.degree(x), reverse=True)

    #Print the top 5 movies in each communities:
    for m_id in community_movie_sorted[:5]:
        title = movie_id_map.get(m_id, f'Unknown movie {m_id}')
        genre = movie_genre_map.get(m_id, f'Unknown genre {m_id}')
        print(f'-- Title: {title} -- Genre: {genre}')


--- Taste Community 1 (5638 movies)
-- Title: Godfather, The (1972) -- Genre: Crime|Drama
-- Title: Pulp Fiction (1994) -- Genre: Comedy|Crime|Drama|Thriller
-- Title: Shawshank Redemption, The (1994) -- Genre: Crime|Drama
-- Title: Lord of the Rings: The Fellowship of the Ring, The (2001) -- Genre: Adventure|Fantasy
-- Title: Star Wars: Episode V - The Empire Strikes Back (1980) -- Genre: Action|Adventure|Sci-Fi

--- Taste Community 2 (2091 movies)
-- Title: Great Escape, The (1963) -- Genre: Action|Adventure|Drama|War
-- Title: Dallas Buyers Club (2013) -- Genre: Drama
-- Title: Jackie Brown (1997) -- Genre: Crime|Drama|Thriller
-- Title: Star Trek II: The Wrath of Khan (1982) -- Genre: Action|Adventure|Sci-Fi|Thriller
-- Title: Sleeper (1973) -- Genre: Comedy|Sci-Fi

--- Taste Community 3 (1723 movies)
-- Title: 2001: A Space Odyssey (1968) -- Genre: Adventure|Drama|Sci-Fi
-- Title: Casablanca (1942) -- Genre: Drama|Romance
-- Title: Dr. Strangelove or: How I Learned to Stop Worryi

Based on the top 5 movie and genres in each of the communities, we can see that **community 1** has a taste mostly for Crime, Drama, and action.

**Community 2** has a taste for Crime, Drama, with a mix of comedy and mystery.

**Community 3** prefers Drama, Romance, and comedy.

**Community 4** has a taste for Horror, Fantasy, and Action.

**Community 5** has a taste for Animation, Children, and adventure.



### 6) Who are the "hub" users?
#### Hub users - users who act as bridges that connect other movie islands together.

In [22]:
# project users to a subgraph to see which user share similar taste in movies:
user_nodes = [n for n in island1 if n.startswith('u_')]
user_network = bipartite.projected_graph(strong_islands, user_nodes)

#see which user share the most rating to the same movie as others:
top_share_user = sorted(user_network.degree, key=lambda x:x[1], reverse=True)

print("Users most frequently co-rated movies with other users:")
print(top_share_user[:10])


Users most frequently co-rated movies with other users:
[('u_286707', 1156), ('u_125273', 1139), ('u_175746', 1119), ('u_306942', 1114), ('u_346366', 1111), ('u_357819', 1110), ('u_314510', 1105), ('u_357370', 1105), ('u_271560', 1103), ('u_90926', 1099)]


These ten users are the main 'hubs' for the largest movie community island. They have the taste for most of the mainstream movie genres that are highly rated by movie goers.


### 7) Top movie genre/choices among the hub users.
#### In other words, what are the mainstream movies that are rated highly in general.

In [23]:
top_user_id = [int(userid.replace('u_', '')) for userid, count in top_share_user[:10]]

#Create a chart to how the top 10 movies choices among the mainstream movie viewers
top_user_favorites = (
    final_df
    .filter(
        (pl.col('userId').is_in(top_user_id)) &
        (pl.col('rating') == 5.0)
    )
    .join(movie_df, on='movieId')
    .group_by(['movieId', 'title', 'genres'])
    .agg(pl.count('userId').alias('overlap_count'))
    .sort('overlap_count', descending=True)
)

display(top_user_favorites.head(10))


movieId,title,genres,overlap_count
i64,str,str,u32
4973,"""Amelie (Fabuleux destin d'Amél…","""Comedy|Romance""",10
4226,"""Memento (2000)""","""Mystery|Thriller""",10
4993,"""Lord of the Rings: The Fellows…","""Adventure|Fantasy""",10
79702,"""Scott Pilgrim vs. the World (2…","""Action|Comedy|Fantasy|Musical|…",10
356,"""Forrest Gump (1994)""","""Comedy|Drama|Romance|War""",9
148626,"""Big Short, The (2015)""","""Drama""",9
68157,"""Inglourious Basterds (2009)""","""Action|Drama|War""",9
3114,"""Toy Story 2 (1999)""","""Adventure|Animation|Children|C…",9
1206,"""Clockwork Orange, A (1971)""","""Crime|Drama|Sci-Fi|Thriller""",9


These movies are so closely ranked by their number of overlaps in ratings 

### 8) To determine the robustness of this network by removing the hub users:

In [24]:
#What happens if we remove these hub users?
hub_user= [user for user, count in top_share_user[:10]]

des_net = strong_islands.copy()
print(f'original number of nodes: {len(des_net.nodes)}')
des_net.remove_nodes_from(hub_user)
print(f'After removal - number of nodes: {len(des_net.nodes)}')
print('-'*80)

# Fix: Convert the generator to a list immediately so it can be reused
new_islands = list(nx.connected_components(des_net))

print(f'Original number of islands: {len(islands)}')
print(f'New number of islands: {len(new_islands)}')
print('-'*80)

#See how much did the main island get reduced after removing the hub users:
print(f'Main island node size: {len(max(islands, key=len))}')
print(f'Main island node size after removal: {len(max(new_islands, key=len))}')

original number of nodes: 11259
After removal - number of nodes: 11249
--------------------------------------------------------------------------------
Original number of islands: 2
New number of islands: 923
--------------------------------------------------------------------------------
Main island node size: 11257
Main island node size after removal: 10326


After the removal of the 10 hub users, the original main island was broken up dramatically.
From merely two islands, the number of islands became over 900, representing a major fragility in the network.
These 10 hub users are the main connectors that bridge the gaps for many other minor or niche movies islands.
On the other hand, the size of the main island only reduced roughly 10%, indicating that the core genres of the movie community are still robustly represented in the network.

From these data, we can infer that these 10 hub users are either professional reviewers or passionate movie goers who can help the rest of the community to explore other types of movie genres. As movie producers, identifying these hub users and have them review the movies can undoubtly help to disseminate the movie to the community a lot quicker.